# Export to a catalog (LSP)
Send an instance and its work to an institution's library services platform (FOLIO, Alma) with `/export/`, and follow the export through Airflow: `monitor_institutions_exports` passes it to the institution's DAG (`stanford`, `cornell`, `penn`, `ucdavis`, …), picked by your Keycloak user's group.

**Exporting writes to a catalog.** If your local Airflow has an institution's catalog credentials, a real export reaches that catalog, so the export itself is off until you set `SEND = True`. The rest only reads. Local stack only.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

In [ ]:
bc.require_writable("Exporting")
AUTH = bc.auth_headers()
s, AIRFLOW = bc.airflow()

## Recent exports
The latest runs of the monitor DAG and of each institution's DAG.

In [ ]:
import pandas as pd

DAGS = ["monitor_institutions_exports", "stanford", "cornell", "penn", "ucdavis"]
rows = []
for dag in DAGS:
    r = s.get(f"{AIRFLOW}/dags/{dag}/dagRuns", params={"limit": 5, "order_by": "-start_date"})
    for run in r.json().get("dag_runs", []) if r.ok else []:
        rows.append({"dag": dag, "run": run["dag_run_id"], "state": run["state"], "start": run.get("start_date"), "resource": (run.get("conf") or {}).get("resource")})
pd.DataFrame(rows).sort_values("start", ascending=False) if rows else "No export runs yet."

## Export one instance
The first instance in search, or set `INSTANCE_URI`. `local_id` overlays an existing catalog record (its HRID, for example) instead of creating one.

In [ ]:
INSTANCE_URI = ""
LOCAL_ID = None
SEND = False

if not INSTANCE_URI:
    uuid_ = bc.sample("instances")
    INSTANCE_URI = bc.get(f"/api/instances/{uuid_}.jsonld").json()["@id"] if uuid_ else ""
print("Instance:", INSTANCE_URI or "none in this stack: create one with the Create, edit and delete records starter")

if SEND and INSTANCE_URI:
    r = bc.post("/api/export/", json={"instance_uri": INSTANCE_URI, "local_id": LOCAL_ID}, headers=AUTH)
    show(r)
    export = r.json()
else:
    print("Not sent. Set SEND = True to export.")
    show(bc.post("/api/export/", json={}, headers=AUTH), 300)   # an empty body is refused (422): checks the route and your login

## Follow it

In [ ]:
if SEND and INSTANCE_URI:
    state, tasks = bc.wait_for_run("monitor_institutions_exports", export["workflow_id"], timeout=600)
    print("monitor_institutions_exports →", state)
    display(pd.DataFrame(tasks))
else:
    print("Nothing sent.")